## tl;dr

The official file passed MD5, schema, shape, label, and count checks for **10,015 images**. The audit found **0 exact cross-split clusters** and applies parallel leakage-aware test reporting.

## Context & Methods

This WP2 notebook reruns the saved audit at one-image-per-record grain.

### Key Assumptions

Difference-hash equality identifies review candidates, not confirmed duplicates.

In [1]:
from pathlib import Path
import json
import derma_agent as project
config = project.load_config()
print(config['data']['dataset_id'], project.DATA_PATH)

dermamnist-v2-28 C:\Users\messl\Downloads\Derma\DermaAgent\main\data\dermamnist.npz


## Data

Load the official checksummed NPZ and show its fixed partition shapes.

In [2]:
arrays = project.load_npz(project.download_dataset(config))
for split in ('train', 'val', 'test'):
    print(split, arrays[f'{split}_images'].shape, arrays[f'{split}_labels'].shape)

train (7007, 28, 28, 3) (7007, 1)
val (1003, 28, 28, 3) (1003, 1)
test (2005, 28, 28, 3) (2005, 1)


## Results

Recompute the audit and reconcile its highest-impact counts to saved D2.1/D2.2 evidence.

In [3]:
registry, audit = project.audit_dataset(config, write_outputs=False)
saved_registry = project.read_json(project.deliverable_path('D2.1'))
saved_audit = project.read_json(project.deliverable_path('D2.2'))
assert registry['sample_count'] == saved_registry['sample_count'] == 10015
for check in ('exact_duplicates', 'cross_split_exact_leakage', 'perceptual_hash_candidates'):
    assert audit['checks'][check]['cluster_count'] == saved_audit['checks'][check]['cluster_count']
print(json.dumps({k: {'clusters': audit['checks'][k]['cluster_count'], 'records': audit['checks'][k]['affected_records']} for k in ('exact_duplicates', 'cross_split_exact_leakage', 'perceptual_hash_candidates')}, indent=2))

{
  "exact_duplicates": {
    "clusters": 2,
    "records": 4
  },
  "cross_split_exact_leakage": {
    "clusters": 0,
    "records": 0
  },
  "perceptual_hash_candidates": {
    "clusters": 9,
    "records": 21
  }
}


## Takeaways

- The official benchmark split remains unchanged for comparability.
- Test images with exact byte matches in train/validation are excluded in the parallel leakage-aware report.
- Perceptual-hash candidates remain visible for review and are not silently removed.
- Training, calibration, uncertainty selection, and model choice never use test labels.